# Trung bình của trung bình: doanh số nửa cuối 2014 giảm 13% hay tăng 3%?

**Bối cảnh.** Họp quý của chuỗi Rossmann. Finance chiếu dashboard: *"Doanh số nửa cuối 2014 giảm 13.2% so với
cùng kỳ 2013."* Head of Ops phản bác: *"Doanh số trung bình một cửa hàng mỗi ngày tăng 3.4%."* CEO hỏi bạn: **ai
đúng, và con số nào nên nằm trong báo cáo gửi hội đồng quản trị?** Sau đó CEO hỏi thêm: *"Cửa hàng loại b bán
nhiều gấp rưỡi loại a, vì sao? Và mỗi khách mang lại bao nhiêu tiền ở từng loại cửa hàng?"*

Dữ liệu: bộ `store-sales` (Rossmann Store Sales, Kaggle). `train.csv` có một dòng cho mỗi cửa hàng mỗi ngày từ
01/01/2013 tới 31/07/2015 (`Store`, `Date`, `Sales`, `Customers`, `Open`, `Promo`, ...), kể cả ngày đóng cửa.
`store.csv` mô tả 1 115 cửa hàng (`StoreType` a/b/c/d, `Assortment`, `CompetitionDistance`, ...).

Việc của bạn: viết bốn hàm để tính trung bình **đúng đơn vị** và **đúng tập đơn vị**, trả lời CEO, rồi **làm phần
trắc nghiệm trong app**.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, vẽ và in kết quả; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/07-descriptive-eda/exercises/intermediate/02-average-of-averages -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

STORE = ensure("store-sales")
train = pd.read_csv(STORE / "train.csv", dtype={"StateHoliday": str}, parse_dates=["Date"])
stores = pd.read_csv(STORE / "store.csv")
print(train.shape, train.Date.min(), "→", train.Date.max(), "|", stores.shape)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Ngày bán hàng và độ phủ dữ liệu

`open_days(train) -> DataFrame`: bản sao các dòng `Open == 1` **và** `Sales > 0` (có vài dòng ghi mở cửa nhưng
doanh số 0, coi là không bán).

`full_coverage_stores(train, start, end) -> list[int]`: danh sách cửa hàng (sắp tăng) có **ít nhất một dòng cho mọi
ngày lịch** trong `[start, end]` (tính cả hai đầu; ngày đóng cửa vẫn có dòng trong `train.csv`). Cột `Date` có thể
là chuỗi hoặc datetime. Đây là **balanced panel**: tập cửa hàng so sánh được giữa các kỳ trong khoảng đó.

In [ ]:
def open_days(train: pd.DataFrame) -> pd.DataFrame:
    """Các dòng Open == 1 và Sales > 0 (bản sao)."""
    return train[(train["Open"] == 1) & (train["Sales"] > 0)].copy()


def full_coverage_stores(train: pd.DataFrame, start: str, end: str) -> list[int]:
    """Cửa hàng có dòng cho mọi ngày lịch trong [start, end], sắp tăng."""
    dates = pd.to_datetime(train["Date"])
    need = pd.date_range(start, end, freq="D")
    inside = dates.between(need[0], need[-1])
    days = dates[inside].groupby(train.loc[inside, "Store"]).nunique()
    return sorted(int(s) for s in days[days == len(need)].index)

In [ ]:
# explore: mỗi tháng có bao nhiêu cửa hàng có dữ liệu?
per_month = train.groupby(train.Date.dt.to_period("M")).Store.nunique()
print(per_month[per_month < 1115])
whole = full_coverage_stores(train, "2013-01-01", "2015-07-31")
h2 = full_coverage_stores(train, "2013-07-01", "2014-12-31")
print(len(whole), "cửa hàng đủ dữ liệu cả giai đoạn;", len(h2), "cửa hàng đủ dữ liệu 07/2013-12/2014")
missing = sorted(set(stores.Store) - set(h2))
print(len(missing), "cửa hàng thiếu; theo loại:", stores[stores.Store.isin(missing)].StoreType.value_counts().to_dict())

## 2. Trung bình theo dòng và trung bình của trung bình

`mean_two_ways(df, by, value="Sales", unit="Store") -> DataFrame`, index là các giá trị của `by` (tên `by`), các
cột đúng thứ tự:

| Cột | Ý nghĩa |
| --- | --- |
| `n_rows` | số dòng (cửa hàng-ngày) của nhóm |
| `n_units` | số cửa hàng khác nhau của nhóm |
| `row_mean` | mean `value` trên mọi dòng của nhóm: mỗi cửa hàng-ngày một phiếu |
| `unit_mean` | mean của mean từng cửa hàng trong nhóm: mỗi cửa hàng một phiếu |

Hai con số trả lời hai câu hỏi khác nhau: "một ngày bán hàng trung bình" và "một cửa hàng trung bình". Chúng
khác nhau khi các cửa hàng có số ngày rất khác nhau.

In [ ]:
def mean_two_ways(df: pd.DataFrame, by: str, value: str = "Sales", unit: str = "Store") -> pd.DataFrame:
    """n_rows, n_units, row_mean (mỗi dòng một phiếu), unit_mean (mỗi đơn vị một phiếu) theo by."""
    g = df.groupby(by)
    per_unit = df.groupby([by, unit])[value].mean()
    out = pd.DataFrame({
        "n_rows": g.size(), "n_units": g[unit].nunique(), "row_mean": g[value].mean(),
        "unit_mean": per_unit.groupby(level=0).mean(),
    })
    out.index.name = by
    return out

## 3. Tỉ số: tổng/tổng hay trung bình các tỉ số?

`ratio_two_ways(df, by, num="Sales", den="Customers") -> DataFrame`, chỉ dùng dòng có `den > 0`; index tên `by`, hai
cột đúng thứ tự:

- `pooled`: tổng `num` / tổng `den` của nhóm;
- `mean_of_ratios`: mean của `num / den` từng dòng.

In [ ]:
def ratio_two_ways(df: pd.DataFrame, by: str, num: str = "Sales", den: str = "Customers") -> pd.DataFrame:
    """pooled = sum(num) / sum(den); mean_of_ratios = mean(num / den); chỉ dòng den > 0."""
    d = df[df[den] > 0]
    g = d.groupby(by)
    out = pd.DataFrame({
        "pooled": g[num].sum() / g[den].sum(),
        "mean_of_ratios": (d[num] / d[den]).groupby(d[by]).mean(),
    })
    out.index.name = by
    return out

## 4. Câu hỏi của CEO, phần 1: nửa cuối 2014

So nửa cuối 2014 (01/07–31/12) với nửa cuối 2013 theo bốn cách: tổng và mean theo dòng, trên mọi cửa hàng có dữ
liệu và trên balanced panel.

In [ ]:
# explore: bốn cách so nửa cuối năm
days = open_days(train)
days["period"] = np.select(
    [days.Date.between("2013-07-01", "2013-12-31"), days.Date.between("2014-07-01", "2014-12-31")],
    ["2013H2", "2014H2"], default="khác",
)
half = days[days.period != "khác"]
panel = half[half.Store.isin(h2)]
rows = {}
for name, d in [("mọi cửa hàng", half), (f"{len(h2)} cửa hàng đủ hai kỳ", panel)]:
    m = mean_two_ways(d, "period")
    tot = d.groupby("period").Sales.sum()
    rows[name] = {
        "cửa hàng 2013H2": m.loc["2013H2", "n_units"], "cửa hàng 2014H2": m.loc["2014H2", "n_units"],
        "tổng 2013H2 (triệu)": tot["2013H2"] / 1e6, "tổng 2014H2 (triệu)": tot["2014H2"] / 1e6,
        "tổng YoY": tot["2014H2"] / tot["2013H2"] - 1,
        "TB/ngày 2013H2": m.loc["2013H2", "row_mean"], "TB/ngày 2014H2": m.loc["2014H2", "row_mean"],
        "TB/ngày YoY": m.loc["2014H2", "row_mean"] / m.loc["2013H2", "row_mean"] - 1,
    }
print(pd.DataFrame(rows).round(3))
gone = half[(half.period == "2013H2")].assign(missing=lambda d: ~d.Store.isin(h2))
print("2013H2, TB/ngày của cửa hàng sẽ vắng mặt vs còn lại:", gone.groupby("missing").Sales.mean().round(0).to_dict())

In [ ]:
# explore: doanh số theo tháng: mọi cửa hàng và balanced panel, kèm số cửa hàng có dữ liệu
monthly = days.groupby(days.Date.dt.to_period("M")).agg(total=("Sales", "sum"), stores=("Store", "nunique"))
monthly_panel = days[days.Store.isin(h2)].groupby(days.Date.dt.to_period("M")).Sales.sum()
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True, height_ratios=[3, 1])
x = monthly.index.to_timestamp()
ax[0].plot(x, monthly.total / 1e6, label="mọi cửa hàng có dữ liệu")
ax[0].plot(x, monthly_panel.reindex(monthly.index).to_numpy() / 1e6, label=f"{len(h2)} cửa hàng đủ dữ liệu")
ax[0].set_ylabel("triệu EUR")
ax[0].legend(frameon=False)
ax[0].set_title("Doanh số tháng: 'cú giảm' nửa cuối 2014 chỉ có ở đường mọi cửa hàng", loc="left")
ax[1].bar(x, monthly.stores, width=20, color="#9aa5b1")
ax[1].set_ylabel("số cửa hàng")
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## 5. Câu hỏi của CEO, phần 2: loại cửa hàng

In [ ]:
# explore: theo loại cửa hàng: hai cách lấy trung bình, và doanh số mỗi khách
by_type = days.merge(stores[["Store", "StoreType"]], on="Store")
t = mean_two_ways(by_type, "StoreType").join(ratio_two_ways(by_type, "StoreType"))
t["ngày/cửa hàng"] = t.n_rows / t.n_units
t["khách TB/ngày"] = by_type.groupby("StoreType").Customers.mean()
t["tỉ lệ ngày Chủ nhật"] = by_type.groupby("StoreType").DayOfWeek.apply(lambda d: (d == 7).mean())
print(t.round(3))
print("chung:", ratio_two_ways(by_type.assign(all="chuỗi"), "all").round(3).to_dict("records"))

Viết ra (cho chính bạn, trước khi làm trắc nghiệm):

1. Một câu trả lời CEO: con số nào đúng cho nửa cuối 2014, và vì sao hai con số trong họp đều sai.
2. Cửa hàng loại b "bán gấp rưỡi" vì bán đắt hơn hay vì đông khách hơn?
3. Dashboard của Finance nên sửa thế nào để chuyện này không lặp lại?

Xong thì mở app, làm phần **trắc nghiệm** của bài.